# Tennis Ball Trajectory Optimization Lab

This notebook is a small sandbox for tuning tennis-ball trajectory post-processing without running the full video pipeline every time.

Recommended loop:
1. Load ball pixels from an existing `match_frames*.json`.
2. Inspect missing frames, speed spikes, and acceleration spikes.
3. Compare the current pipeline-style trajectory with a tunable optimized trajectory.
4. Compare bounce candidates before pushing changes back into `BallTrackerManager`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from dataclasses import dataclass
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"scripts", "experiments"} and PROJECT_ROOT.parent.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cv.config import INPUT_DIR, OUTPUT_DIR as CV_OUTPUT_DIR, SAMPLES_DIR

MPLCONFIGDIR = CV_OUTPUT_DIR / ".matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
XDG_CACHE_HOME = CV_OUTPUT_DIR / ".cache"
XDG_CACHE_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(XDG_CACHE_HOME))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

from cv.tracking.bounce import BounceDetectorManager
from cv.tracking.trajectory import interpolation, remove_outliers_improved, split_track_improved

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 40)


## 1. Pick Input Data

Use `match_frames_v2.json` / `match_frames.json` when you want to tune from an existing run. The notebook reads `frames[*].ball.pixel` and treats missing ball entries as gaps.

In [ ]:
MATCH_FRAMES_PATH = CV_OUTPUT_DIR / "real02_ball_coords_raw.json"
if not MATCH_FRAMES_PATH.exists():
    raise FileNotFoundError(f"Trajectory input not found: {MATCH_FRAMES_PATH}")

with MATCH_FRAMES_PATH.open("r", encoding="utf-8") as handle:
    payload = json.load(handle)

metadata = payload.get("metadata", {})
if "frames" in payload:
    frames = payload["frames"]
elif "ball_coords" in payload:
    frames = [
        {
            "frame_index": item["frame_index"],
            "time_sec": item.get("time_sec"),
            "ball": {
                "pixel": item["pixel"],
                "court": None,
                "confidence": item.get("confidence"),
                "heat_conf": item.get("heat_conf"),
                "peak_class_probability": item.get("peak_class_probability"),
                "predicted_intensity": item.get("predicted_intensity"),
            } if item.get("pixel") else None,
            "players": [],
            "events": {"is_bounce": False, "is_hit": False, "is_net_hit": False},
        }
        for item in payload["ball_coords"]
    ]
else:
    raise ValueError("Unsupported trajectory input JSON. Expected 'frames' or 'ball_coords'.")
fps = float(metadata.get("fps") or 25.0)
frame_width = int(metadata.get("video_width") or 1920)
frame_height = int(metadata.get("video_height") or 1080)

raw_coords = []
raw_confidences = []
event_rows = []
for frame in frames:
    ball = frame.get("ball")
    raw_coords.append(tuple(ball["pixel"]) if isinstance(ball, dict) and ball.get("pixel") else None)
    confidence = None
    if isinstance(ball, dict):
        confidence = ball.get("confidence", ball.get("heat_conf"))
    raw_confidences.append(float(confidence) if confidence is not None else np.nan)
    events = frame.get("events") or {}
    if events.get("is_hit") or events.get("is_bounce") or events.get("is_net_hit"):
        event_rows.append({
            "frame": frame["frame_index"],
            "hit": bool(events.get("is_hit")),
            "bounce": bool(events.get("is_bounce")),
            "net": bool(events.get("is_net_hit")),
            "hitter_id": events.get("hitter_id"),
            "shot_type": events.get("shot_type"),
        })

events_df = pd.DataFrame(event_rows)
print(f"Loaded {len(raw_coords):,} frames from {MATCH_FRAMES_PATH.name}")
print(f"fps={fps}, size={frame_width}x{frame_height}, ball detections={sum(p is not None for p in raw_coords):,}")
events_df.head(20)

## 2. Diagnostics Helpers

In [ ]:
def coords_to_array(coords):
    arr = np.full((len(coords), 2), np.nan, dtype=np.float64)
    for i, point in enumerate(coords):
        if point is None:
            continue
        x, y = point
        if x is None or y is None:
            continue
        arr[i] = [float(x), float(y)]
    return arr


def array_to_coords(arr):
    coords = []
    for x, y in np.asarray(arr, dtype=np.float64):
        if not np.isfinite(x) or not np.isfinite(y):
            coords.append((np.nan, np.nan))
        else:
            coords.append((float(x), float(y)))
    return coords


def valid_mask(arr):
    return np.isfinite(arr[:, 0]) & np.isfinite(arr[:, 1])


def trajectory_stats(name, coords):
    arr = coords_to_array(coords)
    mask = valid_mask(arr)
    velocity = np.linalg.norm(np.diff(arr, axis=0), axis=1)
    acceleration = np.abs(np.diff(velocity))
    valid_velocity = velocity[np.isfinite(velocity)]
    valid_acceleration = acceleration[np.isfinite(acceleration)]
    return {
        "name": name,
        "frames": len(arr),
        "valid": int(mask.sum()),
        "valid_ratio": round(float(mask.mean()), 4),
        "median_speed_px_frame": round(float(np.nanmedian(valid_velocity)), 2) if len(valid_velocity) else None,
        "p95_speed_px_frame": round(float(np.nanpercentile(valid_velocity, 95)), 2) if len(valid_velocity) else None,
        "p99_speed_px_frame": round(float(np.nanpercentile(valid_velocity, 99)), 2) if len(valid_velocity) else None,
        "p95_accel_px_frame2": round(float(np.nanpercentile(valid_acceleration, 95)), 2) if len(valid_acceleration) else None,
    }


def clamp_window(start, end, length):
    if length <= 0:
        return 0, 0
    safe_start = max(0, min(int(start), length - 1))
    safe_end = max(safe_start + 1, min(int(end), length))
    return safe_start, safe_end


def plot_window(named_coords, start=0, end=300, events=None, title="Trajectory window"):
    max_len = max((len(coords) for coords in named_coords.values()), default=0)
    safe_start, safe_end = clamp_window(start, end, max_len)
    if safe_end <= safe_start:
        print(f"No frames available for requested window {start}:{end}; data length={max_len}")
        return

    if safe_start != start or safe_end != end:
        print(f"Requested window {start}:{end} was clamped to {safe_start}:{safe_end} for data length={max_len}.")

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    for name, coords in named_coords.items():
        full_arr = coords_to_array(coords)
        local_start, local_end = clamp_window(safe_start, safe_end, len(full_arr))
        arr = full_arr[local_start:local_end]
        xs = np.arange(local_start, local_end)
        axes[0].plot(xs, arr[:, 0], marker=".", linewidth=1, markersize=2, label=name)
        axes[1].plot(xs, arr[:, 1], marker=".", linewidth=1, markersize=2, label=name)

    if events is not None and not events.empty:
        for _, row in events[(events.frame >= safe_start) & (events.frame < safe_end)].iterrows():
            color = "tab:red" if row.hit else "tab:green" if row.bounce else "tab:purple"
            for axis in axes:
                axis.axvline(row.frame, color=color, alpha=0.25, linewidth=1)

    axes[0].set_ylabel("x pixel")
    axes[1].set_ylabel("y pixel")
    axes[1].set_xlabel("frame")
    axes[0].set_title(title)
    axes[0].legend(loc="upper right")
    axes[1].legend(loc="upper right")
    plt.show()


def plot_court_projection(named_coords, start=0, end=None, invert_y=True):
    max_len = max((len(coords) for coords in named_coords.values()), default=0)
    requested_end = max_len if end is None else end
    safe_start, safe_end = clamp_window(start, requested_end, max_len)
    if safe_start != start or safe_end != requested_end:
        print(f"Requested projection window {start}:{requested_end} was clamped to {safe_start}:{safe_end} for data length={max_len}.")
    plt.figure(figsize=(10, 7))
    for name, coords in named_coords.items():
        full_arr = coords_to_array(coords)
        local_start, local_end = clamp_window(safe_start, safe_end, len(full_arr))
        arr = full_arr[local_start:local_end]
        plt.plot(arr[:, 0], arr[:, 1], marker=".", linewidth=0.8, markersize=2, label=name)
    if invert_y:
        plt.gca().invert_yaxis()
    plt.xlim(0, frame_width)
    plt.ylim(frame_height, 0) if invert_y else plt.ylim(0, frame_height)
    plt.xlabel("x pixel")
    plt.ylabel("y pixel")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


pd.DataFrame([trajectory_stats("raw", raw_coords)])

## 3. Reproduce Current Pipeline-Style Processing

This mirrors the current `BallTrackerManager.process_trajectory(..., enable_smoothing=False)` shape: fixed outlier thresholds, split by long missing gaps, then interpolate inside each segment.

In [ ]:
def current_pipeline_process(raw_coords, *, v_threshold=20, a_threshold=20, max_gap=10, enable_smoothing=False):
    cleaned = [point if point is not None else (np.nan, np.nan) for point in raw_coords]
    remove_outliers_improved(cleaned, v_threshold=v_threshold, a_threshold=a_threshold)
    segments = split_track_improved(cleaned, max_gap=max_gap)
    final = [(np.nan, np.nan)] * len(cleaned)

    for start, end in segments:
        section = [point[:] if point is not None else None for point in cleaned[start:end + 1]]
        interpolated = interpolation(section)
        arr = np.array(interpolated, dtype=np.float64)
        if enable_smoothing and len(arr) >= 7:
            arr[:, 0] = savgol_filter(arr[:, 0], window_length=7, polyorder=2)
            arr[:, 1] = savgol_filter(arr[:, 1], window_length=7, polyorder=2)
        final[start:end + 1] = array_to_coords(arr)
    return final, segments


current_coords, current_segments = current_pipeline_process(raw_coords)
pd.DataFrame([
    trajectory_stats("raw", raw_coords),
    trajectory_stats("current", current_coords),
])

## 4. Tunable Optimized Processing

Idea: use robust local motion thresholds instead of a fixed 20 px/frame cutoff. Keep a cleaned/interpolated track for event detection, and a smoothed track for visualization/speed-like features.

In [ ]:
@dataclass
class TrajectoryConfig:
    max_gap: int = 10
    min_segment_len: int = 5
    speed_mad_scale: float = 6.0
    accel_mad_scale: float = 7.0
    min_speed_threshold: float = 28.0
    max_speed_threshold: float = 180.0
    smooth_window: int = 7
    smooth_polyorder: int = 2
    preserve_turning_points: bool = True


def robust_threshold(values, *, mad_scale, lower, upper):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return lower
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    robust_sigma = 1.4826 * mad
    threshold = median + mad_scale * robust_sigma
    return float(np.clip(threshold, lower, upper))


def split_valid_segments(arr, max_gap):
    coords = array_to_coords(arr)
    return split_track_improved(coords, max_gap=max_gap)


def adaptive_remove_outliers(raw_coords, config: TrajectoryConfig):
    arr = coords_to_array(raw_coords)
    mask = valid_mask(arr)
    speed = np.full(len(arr), np.nan)
    speed[1:] = np.linalg.norm(arr[1:] - arr[:-1], axis=1)
    accel = np.full(len(arr), np.nan)
    accel[2:] = np.abs(speed[2:] - speed[1:-1])

    speed_threshold = robust_threshold(
        speed,
        mad_scale=config.speed_mad_scale,
        lower=config.min_speed_threshold,
        upper=config.max_speed_threshold,
    )
    accel_threshold = robust_threshold(
        accel,
        mad_scale=config.accel_mad_scale,
        lower=config.min_speed_threshold,
        upper=config.max_speed_threshold,
    )

    keep = mask.copy()
    for i in range(1, len(arr) - 1):
        if not keep[i]:
            continue
        prev_ok = keep[i - 1]
        next_ok = mask[i + 1]
        jump_from_prev = prev_ok and np.linalg.norm(arr[i] - arr[i - 1]) > speed_threshold
        jump_to_next = next_ok and np.linalg.norm(arr[i + 1] - arr[i]) > speed_threshold

        if jump_from_prev and jump_to_next:
            bridge = (arr[i - 1] + arr[i + 1]) / 2.0 if prev_ok and next_ok else None
            bridge_error = np.linalg.norm(arr[i] - bridge) if bridge is not None else np.inf
            if bridge_error > speed_threshold * 0.75:
                keep[i] = False
                continue

        if prev_ok and next_ok:
            v1 = arr[i] - arr[i - 1]
            v2 = arr[i + 1] - arr[i]
            local_accel = np.linalg.norm(v2 - v1)
            if local_accel > accel_threshold:
                if not config.preserve_turning_points:
                    keep[i] = False
                else:
                    # Preserve plausible bounce / hit turning points when the point stays close to the local bridge.
                    bridge = (arr[i - 1] + arr[i + 1]) / 2.0
                    if np.linalg.norm(arr[i] - bridge) > speed_threshold:
                        keep[i] = False

    cleaned = arr.copy()
    cleaned[~keep] = np.nan
    info = {
        "speed_threshold": round(speed_threshold, 2),
        "accel_threshold": round(accel_threshold, 2),
        "removed_points": int(mask.sum() - keep.sum()),
    }
    return cleaned, info


def interpolate_segments(cleaned_arr, config: TrajectoryConfig):
    segments = split_valid_segments(cleaned_arr, max_gap=config.max_gap)
    interpolated = np.full_like(cleaned_arr, np.nan)
    for start, end in segments:
        if end - start + 1 < config.min_segment_len:
            continue
        section = array_to_coords(cleaned_arr[start:end + 1])
        interpolated_section = np.array(interpolation(section), dtype=np.float64)
        interpolated[start:end + 1] = interpolated_section
    return interpolated, segments


def smooth_segments(interpolated_arr, segments, config: TrajectoryConfig):
    smoothed = interpolated_arr.copy()
    window = config.smooth_window
    if window % 2 == 0:
        window += 1
    for start, end in segments:
        length = end - start + 1
        if length < window:
            continue
        section = smoothed[start:end + 1]
        if not np.all(np.isfinite(section)):
            continue
        smoothed[start:end + 1, 0] = savgol_filter(section[:, 0], window, config.smooth_polyorder)
        smoothed[start:end + 1, 1] = savgol_filter(section[:, 1], window, config.smooth_polyorder)
    return smoothed


def optimized_process(raw_coords, config=TrajectoryConfig()):
    cleaned, info = adaptive_remove_outliers(raw_coords, config)
    interpolated, segments = interpolate_segments(cleaned, config)
    smoothed = smooth_segments(interpolated, segments, config)
    return {
        "cleaned": array_to_coords(cleaned),
        "event": array_to_coords(interpolated),
        "smooth": array_to_coords(smoothed),
        "segments": segments,
        "info": info,
    }


config = TrajectoryConfig(
    max_gap=10,
    speed_mad_scale=6.0,
    accel_mad_scale=7.0,
    min_speed_threshold=28.0,
    max_speed_threshold=180.0,
    smooth_window=7,
)
optimized = optimized_process(raw_coords, config)
optimized["info"]

In [ ]:
pd.DataFrame([
    trajectory_stats("raw", raw_coords),
    trajectory_stats("current", current_coords),
    trajectory_stats("optimized_event", optimized["event"]),
    trajectory_stats("optimized_smooth", optimized["smooth"]),
])

## 4B. Window RANSAC Processing

This adapts the earlier Window RANSAC experiment into a safer two-track output. The first pass only scores points and groups consecutive suspects. Before correction, independent left/right models test whether a suspect run is a real trajectory breakpoint: nearby extrapolated positions plus a large velocity-angle change preserve the raw run. The second pass excludes each remaining suspect run and its guard frames before fitting, and only corrects a point when the fitted inliers cover both sides of the target frame. `ransac_event` keeps breakpoint and ambiguous raw points, while `ransac_smooth` uses predictions only from trusted fits.

In [ ]:
@dataclass
class WindowRansacConfig:
    window_half: int = 10
    suspect_context_half: int = 20
    ransac_iters: int = 80
    base_inlier_thresh: float = 16.0
    min_points: int = 4
    fit_degree_x: int = 1
    fit_degree_y: int = 2
    max_fill_gap: int = 10
    min_inlier_ratio: float = 0.45
    min_correction_inlier_ratio: float = 0.60
    min_side_inliers: int = 3
    suspect_merge_gap: int = 2
    suspect_guard: int = 1
    breakpoint_min_angle_deg: float = 25.0
    breakpoint_position_scale: float = 2.0
    breakpoint_support_scale: float = 0.65
    breakpoint_min_support_ratio: float = 0.35
    breakpoint_min_side_inliers: int = 4
    breakpoint_min_side_ratio: float = 0.60
    far_thresh_scale: float = 0.65
    near_thresh_scale: float = 2.30
    conf_low: float = 0.50
    conf_high: float = 1.00
    conf_thresh_scale_low: float = 0.75
    conf_thresh_scale_high: float = 1.60
    random_seed: int = 42


def perspective_threshold_scale(point_y, frame_h, config: WindowRansacConfig):
    if frame_h <= 0 or point_y is None or not np.isfinite(point_y):
        return 1.0
    y_ratio = float(np.clip(point_y / frame_h, 0.0, 1.0))
    return config.far_thresh_scale + y_ratio * (config.near_thresh_scale - config.far_thresh_scale)


def confidence_threshold_scale(confidence, config: WindowRansacConfig):
    if confidence is None or not np.isfinite(confidence):
        return 1.0
    ratio = (float(confidence) - config.conf_low) / max(config.conf_high - config.conf_low, 1e-6)
    ratio = float(np.clip(ratio, 0.0, 1.0))
    return config.conf_thresh_scale_low + ratio * (config.conf_thresh_scale_high - config.conf_thresh_scale_low)


def adaptive_ransac_threshold(base_thresh, point_y, confidence, frame_h, config: WindowRansacConfig):
    return base_thresh * perspective_threshold_scale(point_y, frame_h, config) * confidence_threshold_scale(confidence, config)


def collect_ransac_window(raw_arr, confidences, center, config: WindowRansacConfig, excluded_mask=None, window_half=None):
    half = config.window_half if window_half is None else int(window_half)
    start = max(0, center - half)
    end = min(len(raw_arr) - 1, center + half)
    frames_w, xs, ys, thresholds = [], [], [], []
    for frame_index in range(start, end + 1):
        if frame_index == center or (excluded_mask is not None and excluded_mask[frame_index]):
            continue
        x, y = raw_arr[frame_index]
        if not np.isfinite(x) or not np.isfinite(y):
            continue
        confidence = confidences[frame_index] if frame_index < len(confidences) else np.nan
        frames_w.append(frame_index - center)
        xs.append(x)
        ys.append(y)
        thresholds.append(adaptive_ransac_threshold(config.base_inlier_thresh, y, confidence, frame_height, config))
    return frames_w, xs, ys, thresholds


def fit_window_ransac(frames_w, xs, ys, thresholds, config: WindowRansacConfig, *, center_frame: int):
    n_points = len(frames_w)
    if n_points < config.min_points:
        return None, 0, None
    t = np.asarray(frames_w, dtype=np.float64)
    x_arr = np.asarray(xs, dtype=np.float64)
    y_arr = np.asarray(ys, dtype=np.float64)
    threshold_arr = np.asarray(thresholds, dtype=np.float64)
    sample_size = min(n_points, max(config.fit_degree_x + 1, config.fit_degree_y + 1) + 1)
    rng = np.random.default_rng(config.random_seed + center_frame)
    best_inliers = 0
    best_cx = None
    best_cy = None
    best_mask = None
    best_median_residual = np.inf
    for _ in range(config.ransac_iters):
        sample_indices = rng.choice(n_points, size=sample_size, replace=False)
        try:
            cx = np.polyfit(t[sample_indices], x_arr[sample_indices], config.fit_degree_x)
            cy = np.polyfit(t[sample_indices], y_arr[sample_indices], config.fit_degree_y)
        except (np.linalg.LinAlgError, ValueError):
            continue
        residuals = np.hypot(x_arr - np.polyval(cx, t), y_arr - np.polyval(cy, t))
        mask = residuals <= threshold_arr
        inlier_count = int(mask.sum())
        median_residual = float(np.median(residuals[mask])) if inlier_count else np.inf
        is_better = inlier_count > best_inliers or (inlier_count == best_inliers and median_residual < best_median_residual)
        if is_better:
            best_inliers = inlier_count
            best_cx = cx
            best_cy = cy
            best_mask = mask
            best_median_residual = median_residual
    min_fit_points = max(config.fit_degree_x + 1, config.fit_degree_y + 1)
    if best_mask is not None and best_inliers >= min_fit_points:
        try:
            best_cx = np.polyfit(t[best_mask], x_arr[best_mask], config.fit_degree_x)
            best_cy = np.polyfit(t[best_mask], y_arr[best_mask], config.fit_degree_y)
        except (np.linalg.LinAlgError, ValueError):
            pass
    if best_cx is None or best_cy is None:
        return None, 0, None
    return (best_cx, best_cy), best_inliers, best_mask


def predict_ransac(params, relative_t=0.0):
    if params is None:
        return np.array([np.nan, np.nan], dtype=np.float64)
    cx, cy = params
    return np.array([np.polyval(cx, relative_t), np.polyval(cy, relative_t)], dtype=np.float64)


def ransac_fit_quality(frames_w, inlier_mask, config: WindowRansacConfig):
    if inlier_mask is None or len(frames_w) == 0:
        return {"inlier_ratio": 0.0, "left_inliers": 0, "right_inliers": 0, "bilateral": False, "trusted": False}
    relative_frames = np.asarray(frames_w, dtype=np.float64)
    inliers = np.asarray(inlier_mask, dtype=bool)
    inlier_ratio = float(inliers.sum() / max(len(relative_frames), 1))
    left_inliers = int(np.sum(inliers & (relative_frames < 0)))
    right_inliers = int(np.sum(inliers & (relative_frames > 0)))
    bilateral = left_inliers >= config.min_side_inliers and right_inliers >= config.min_side_inliers
    trusted = bilateral and inlier_ratio >= config.min_correction_inlier_ratio
    return {
        "inlier_ratio": inlier_ratio,
        "left_inliers": left_inliers,
        "right_inliers": right_inliers,
        "bilateral": bilateral,
        "trusted": trusted,
    }


def merge_suspect_runs(suspect_mask, max_gap):
    merged = np.zeros(len(suspect_mask), dtype=bool)
    indices = np.flatnonzero(suspect_mask)
    if len(indices) == 0:
        return merged
    run_start = int(indices[0])
    run_end = run_start
    for index in indices[1:]:
        index = int(index)
        if index - run_end - 1 <= max_gap:
            run_end = index
            continue
        merged[run_start:run_end + 1] = True
        run_start = run_end = index
    merged[run_start:run_end + 1] = True
    return merged


def dilate_mask(mask, radius):
    dilated = np.asarray(mask, dtype=bool).copy()
    if radius <= 0:
        return dilated
    for index in np.flatnonzero(mask):
        start = max(0, int(index) - radius)
        end = min(len(mask), int(index) + radius + 1)
        dilated[start:end] = True
    return dilated


def mask_runs(mask):
    runs = []
    start = None
    for index, active in enumerate(np.asarray(mask, dtype=bool)):
        if active and start is None:
            start = index
        if start is not None and (not active or index == len(mask) - 1):
            end = index if active and index == len(mask) - 1 else index - 1
            runs.append((start, end))
            start = None
    return runs


def predict_ransac_velocity(params, relative_t=0.0):
    if params is None:
        return np.array([np.nan, np.nan], dtype=np.float64)
    cx, cy = params
    return np.array([np.polyval(np.polyder(cx), relative_t), np.polyval(np.polyder(cy), relative_t)], dtype=np.float64)


def fit_ransac_indices(raw_arr, confidences, indices, origin, config: WindowRansacConfig, *, seed_frame):
    frames_w, xs, ys, thresholds = [], [], [], []
    for frame_index in indices:
        x, y = raw_arr[frame_index]
        if not np.isfinite(x) or not np.isfinite(y):
            continue
        confidence = confidences[frame_index] if frame_index < len(confidences) else np.nan
        frames_w.append(frame_index - origin)
        xs.append(x)
        ys.append(y)
        thresholds.append(adaptive_ransac_threshold(config.base_inlier_thresh, y, confidence, frame_height, config))
    params, inlier_count, inlier_mask = fit_window_ransac(
        frames_w, xs, ys, thresholds, config, center_frame=seed_frame
    )
    inlier_ratio = inlier_count / max(len(frames_w), 1)
    return params, inlier_count, inlier_ratio


def detect_breakpoint_runs(raw_arr, confidences, suspect_mask, config: WindowRansacConfig):
    protected = np.zeros(len(raw_arr), dtype=bool)
    event_runs = np.zeros(len(raw_arr), dtype=bool)
    breakpoint_frames = np.zeros(len(raw_arr), dtype=bool)
    breakpoint_angles = np.full(len(raw_arr), np.nan, dtype=np.float64)
    breakpoint_position_errors = np.full(len(raw_arr), np.nan, dtype=np.float64)

    for run_start, run_end in mask_runs(suspect_mask):
        left_indices = range(max(0, run_start - config.suspect_context_half), run_start)
        right_indices = range(run_end + 1, min(len(raw_arr), run_end + config.suspect_context_half + 1))
        left_params, left_count, left_ratio = fit_ransac_indices(
            raw_arr, confidences, left_indices, run_start, config, seed_frame=run_start
        )
        right_params, right_count, right_ratio = fit_ransac_indices(
            raw_arr, confidences, right_indices, run_start, config, seed_frame=run_end + len(raw_arr)
        )
        sides_trusted = (
            left_params is not None
            and right_params is not None
            and left_count >= config.breakpoint_min_side_inliers
            and right_count >= config.breakpoint_min_side_inliers
            and left_ratio >= config.breakpoint_min_side_ratio
            and right_ratio >= config.breakpoint_min_side_ratio
        )
        if not sides_trusted:
            continue

        best_candidate = None
        for frame_index in range(run_start, run_end + 1):
            relative_t = frame_index - run_start
            left_position = predict_ransac(left_params, relative_t)
            right_position = predict_ransac(right_params, relative_t)
            left_velocity = predict_ransac_velocity(left_params, relative_t)
            right_velocity = predict_ransac_velocity(right_params, relative_t)
            if not all(np.all(np.isfinite(value)) for value in [left_position, right_position, left_velocity, right_velocity]):
                continue
            left_speed = float(np.linalg.norm(left_velocity))
            right_speed = float(np.linalg.norm(right_velocity))
            if left_speed < 1e-6 or right_speed < 1e-6:
                continue
            cosine = float(np.clip(np.dot(left_velocity, right_velocity) / (left_speed * right_speed), -1.0, 1.0))
            angle_deg = float(np.degrees(np.arccos(cosine)))
            position_error = float(np.linalg.norm(left_position - right_position))
            predicted_y = float((left_position[1] + right_position[1]) / 2.0)
            confidence = confidences[frame_index] if frame_index < len(confidences) else np.nan
            position_threshold = adaptive_ransac_threshold(
                config.base_inlier_thresh, predicted_y, confidence, frame_height, config
            ) * config.breakpoint_position_scale
            if angle_deg < config.breakpoint_min_angle_deg or position_error > position_threshold:
                continue
            candidate_score = (angle_deg, -position_error, -abs(frame_index - (run_start + run_end) / 2.0))
            if best_candidate is None or candidate_score > best_candidate[0]:
                best_candidate = (candidate_score, frame_index, angle_deg, position_error)

        if best_candidate is None:
            continue

        _, model_frame, angle_deg, position_error = best_candidate
        supported_frames = []
        assignments = []
        valid_run_points = 0
        for frame_index in range(run_start, run_end + 1):
            raw_point = raw_arr[frame_index]
            if not np.all(np.isfinite(raw_point)):
                continue
            valid_run_points += 1
            relative_t = frame_index - run_start
            left_position = predict_ransac(left_params, relative_t)
            right_position = predict_ransac(right_params, relative_t)
            left_error = float(np.linalg.norm(raw_point - left_position))
            right_error = float(np.linalg.norm(raw_point - right_position))
            confidence = confidences[frame_index] if frame_index < len(confidences) else np.nan
            support_threshold = adaptive_ransac_threshold(
                config.base_inlier_thresh, raw_point[1], confidence, frame_height, config
            ) * config.breakpoint_support_scale
            support_error = max(left_error, right_error) if run_start == run_end else min(left_error, right_error)
            if support_error <= support_threshold:
                supported_frames.append(frame_index)
                assignments.append(0 if left_error <= right_error else 1)

        min_supported = 1 if valid_run_points <= 1 else max(2, int(np.ceil(valid_run_points * config.breakpoint_min_support_ratio)))
        has_reverse_switch = any(current < previous for previous, current in zip(assignments, assignments[1:]))
        if len(supported_frames) < min_supported or has_reverse_switch:
            continue

        event_runs[run_start:run_end + 1] = True
        protected[supported_frames] = True
        frame_index = min(supported_frames, key=lambda index: abs(index - model_frame))
        breakpoint_frames[frame_index] = True
        breakpoint_angles[frame_index] = angle_deg
        breakpoint_position_errors[frame_index] = position_error

    return protected, event_runs, breakpoint_frames, breakpoint_angles, breakpoint_position_errors


def is_short_gap(raw_mask, frame_index, max_gap):
    previous_indices = np.flatnonzero(raw_mask[:frame_index])
    next_indices = np.flatnonzero(raw_mask[frame_index + 1:])
    if len(previous_indices) == 0 or len(next_indices) == 0:
        return False
    prev_idx = int(previous_indices[-1])
    next_idx = int(next_indices[0] + frame_index + 1)
    return (next_idx - prev_idx - 1) <= max_gap


def window_ransac_process(raw_coords, confidences=None, config=WindowRansacConfig()):
    raw_arr = coords_to_array(raw_coords)
    raw_mask = valid_mask(raw_arr)
    if confidences is None:
        confidences = [np.nan] * len(raw_coords)
    confidences = np.asarray(confidences, dtype=np.float64)

    # Pass 1 only scores the raw track. It never changes points, so consecutive
    # detections cannot become self-confirming inputs to the correction pass.
    initial_residuals = np.full(len(raw_arr), np.nan, dtype=np.float64)
    initial_thresholds = np.full(len(raw_arr), config.base_inlier_thresh, dtype=np.float64)
    for frame_index in range(len(raw_arr)):
        if not raw_mask[frame_index]:
            continue
        frames_w, xs, ys, ths = collect_ransac_window(
            raw_arr, confidences, frame_index, config, window_half=config.suspect_context_half
        )
        params, _, _ = fit_window_ransac(frames_w, xs, ys, ths, config, center_frame=frame_index)
        prediction = predict_ransac(params, 0.0)
        if not np.all(np.isfinite(prediction)):
            continue
        confidence = confidences[frame_index] if frame_index < len(confidences) else np.nan
        threshold = adaptive_ransac_threshold(
            config.base_inlier_thresh, raw_arr[frame_index, 1], confidence, frame_height, config
        )
        initial_thresholds[frame_index] = threshold
        initial_residuals[frame_index] = float(np.linalg.norm(raw_arr[frame_index] - prediction))

    initial_suspects = raw_mask & np.isfinite(initial_residuals) & (initial_residuals > initial_thresholds)
    suspect_mask = merge_suspect_runs(initial_suspects, config.suspect_merge_gap)
    breakpoint_keep_mask, breakpoint_event_run_mask, breakpoint_frames, breakpoint_angles, breakpoint_position_errors = detect_breakpoint_runs(
        raw_arr, confidences, suspect_mask, config
    )
    fit_exclusion_mask = dilate_mask(suspect_mask, config.suspect_guard)

    # Pass 2 excludes the complete suspect runs and only trusts models supported
    # by enough inliers on both sides of the target frame.
    event_arr = np.full_like(raw_arr, np.nan)
    smooth_arr = np.full_like(raw_arr, np.nan)
    statuses = ["gap"] * len(raw_arr)
    residuals = np.full(len(raw_arr), np.nan, dtype=np.float64)
    thresholds = np.full(len(raw_arr), config.base_inlier_thresh, dtype=np.float64)
    inlier_counts = np.zeros(len(raw_arr), dtype=np.int32)
    window_points = np.zeros(len(raw_arr), dtype=np.int32)
    left_inliers = np.zeros(len(raw_arr), dtype=np.int32)
    right_inliers = np.zeros(len(raw_arr), dtype=np.int32)
    trusted_models = np.zeros(len(raw_arr), dtype=bool)
    for frame_index in range(len(raw_arr)):
        fit_window_half = config.suspect_context_half if suspect_mask[frame_index] else config.window_half
        frames_w, xs, ys, ths = collect_ransac_window(
            raw_arr, confidences, frame_index, config, excluded_mask=fit_exclusion_mask, window_half=fit_window_half
        )
        window_points[frame_index] = len(frames_w)
        if len(frames_w) < config.min_points:
            if raw_mask[frame_index]:
                event_arr[frame_index] = raw_arr[frame_index]
                smooth_arr[frame_index] = raw_arr[frame_index]
                statuses[frame_index] = "ambiguous_raw_kept" if suspect_mask[frame_index] else "raw_kept"
            continue
        params, inlier_count, inlier_mask = fit_window_ransac(
            frames_w, xs, ys, ths, config, center_frame=frame_index
        )
        inlier_counts[frame_index] = inlier_count
        quality = ransac_fit_quality(frames_w, inlier_mask, config)
        left_inliers[frame_index] = quality["left_inliers"]
        right_inliers[frame_index] = quality["right_inliers"]
        trusted_models[frame_index] = quality["trusted"]
        prediction = predict_ransac(params, 0.0)
        if not np.all(np.isfinite(prediction)):
            if raw_mask[frame_index]:
                event_arr[frame_index] = raw_arr[frame_index]
                smooth_arr[frame_index] = raw_arr[frame_index]
                statuses[frame_index] = "ambiguous_raw_kept" if suspect_mask[frame_index] else "raw_kept"
            continue
        point_y = raw_arr[frame_index, 1] if raw_mask[frame_index] else prediction[1]
        confidence = confidences[frame_index] if frame_index < len(confidences) else np.nan
        threshold = adaptive_ransac_threshold(config.base_inlier_thresh, point_y, confidence, frame_height, config)
        thresholds[frame_index] = threshold
        if raw_mask[frame_index]:
            residual = float(np.linalg.norm(raw_arr[frame_index] - prediction))
            residuals[frame_index] = residual
            if breakpoint_keep_mask[frame_index]:
                event_arr[frame_index] = raw_arr[frame_index]
                smooth_arr[frame_index] = raw_arr[frame_index]
                statuses[frame_index] = "breakpoint_raw_kept"
            elif not suspect_mask[frame_index]:
                event_arr[frame_index] = raw_arr[frame_index]
                smooth_arr[frame_index] = prediction if quality["trusted"] else raw_arr[frame_index]
                statuses[frame_index] = "inlier_raw_kept"
            elif not quality["trusted"]:
                event_arr[frame_index] = raw_arr[frame_index]
                smooth_arr[frame_index] = raw_arr[frame_index]
                statuses[frame_index] = "ambiguous_raw_kept"
            elif residual <= threshold:
                event_arr[frame_index] = raw_arr[frame_index]
                smooth_arr[frame_index] = prediction
                statuses[frame_index] = "suspect_recovered_raw_kept"
            else:
                event_arr[frame_index] = prediction
                smooth_arr[frame_index] = prediction
                statuses[frame_index] = "outlier_corrected"
        else:
            if breakpoint_event_run_mask[frame_index]:
                statuses[frame_index] = "breakpoint_gap"
                continue
            can_fill = (
                is_short_gap(raw_mask, frame_index, config.max_fill_gap)
                and quality["bilateral"]
                and quality["inlier_ratio"] >= config.min_inlier_ratio
            )
            if can_fill:
                event_arr[frame_index] = prediction
                smooth_arr[frame_index] = prediction
                statuses[frame_index] = "short_gap_interp"
    summary = pd.Series(statuses).value_counts().to_dict()
    info = {
        "summary": summary,
        "initial_suspect_points": int(initial_suspects.sum()),
        "merged_suspect_points": int(suspect_mask.sum()),
        "protected_breakpoint_points": int(breakpoint_keep_mask.sum()),
        "breakpoint_event_run_points": int(breakpoint_event_run_mask.sum()),
        "breakpoint_count": int(breakpoint_frames.sum()),
        "trusted_models": int(trusted_models.sum()),
        "median_residual": round(float(np.nanmedian(residuals)), 2) if np.isfinite(residuals).any() else None,
        "median_threshold": round(float(np.nanmedian(thresholds)), 2),
    }
    return {
        "event": array_to_coords(event_arr),
        "smooth": array_to_coords(smooth_arr),
        "statuses": statuses,
        "residuals": residuals,
        "thresholds": thresholds,
        "initial_residuals": initial_residuals,
        "initial_thresholds": initial_thresholds,
        "suspect_mask": suspect_mask,
        "breakpoint_keep_mask": breakpoint_keep_mask,
        "breakpoint_run_mask": breakpoint_event_run_mask,
        "breakpoint_frames": breakpoint_frames,
        "breakpoint_angles": breakpoint_angles,
        "breakpoint_position_errors": breakpoint_position_errors,
        "fit_exclusion_mask": fit_exclusion_mask,
        "inlier_counts": inlier_counts,
        "window_points": window_points,
        "left_inliers": left_inliers,
        "right_inliers": right_inliers,
        "trusted_models": trusted_models,
        "info": info,
    }


ransac_config = WindowRansacConfig(window_half=10, ransac_iters=80, base_inlier_thresh=16.0, max_fill_gap=10)
ransac = window_ransac_process(raw_coords, raw_confidences, ransac_config)
ransac["info"]

In [ ]:
pd.DataFrame([
    trajectory_stats("raw", raw_coords),
    trajectory_stats("current", current_coords),
    trajectory_stats("optimized_event", optimized["event"]),
    trajectory_stats("optimized_smooth", optimized["smooth"]),
    trajectory_stats("ransac_event", ransac["event"]),
    trajectory_stats("ransac_smooth", ransac["smooth"]),
])

In [ ]:
status_df = pd.DataFrame({
    "frame": np.arange(len(raw_coords)),
    "status": ransac["statuses"],
    "initial_residual_px": ransac["initial_residuals"],
    "residual_px": ransac["residuals"],
    "threshold_px": ransac["thresholds"],
    "suspect": ransac["suspect_mask"],
    "breakpoint_protected": ransac["breakpoint_keep_mask"],
    "breakpoint_event_run": ransac["breakpoint_run_mask"],
    "breakpoint_frame": ransac["breakpoint_frames"],
    "breakpoint_angle_deg": ransac["breakpoint_angles"],
    "trusted_model": ransac["trusted_models"],
    "inlier_count": ransac["inlier_counts"],
    "left_inliers": ransac["left_inliers"],
    "right_inliers": ransac["right_inliers"],
    "window_points": ransac["window_points"],
})
status_df[status_df.status.isin(["breakpoint_raw_kept", "outlier_corrected", "ambiguous_raw_kept", "short_gap_interp"])].head(30)

## 5. Visual Compare

Change `START` and `END` to inspect suspicious windows. Red lines are hits, green lines are bounces, purple lines are net hits from the loaded JSON. The next cell also saves a full-length `x(frame)` / `y(frame)` comparison PNG to `cv/data/output`.

In [ ]:
WINDOW_SIZE = min(120, len(raw_coords))
START = max(0, min(650, max(len(raw_coords) - WINDOW_SIZE, 0)))
END = min(len(raw_coords), START + WINDOW_SIZE)

plot_window(
    {
        "raw": raw_coords,
        "current": current_coords,
        "optimized_event": optimized["event"],
        "optimized_smooth": optimized["smooth"],
        "ransac_event": ransac["event"],
        "ransac_smooth": ransac["smooth"],
    },
    START,
    END,
    events_df,
    title=f"Trajectory compare: frames {START}-{END}",
)


def save_xy_function_plot(raw_coords, ransac_result, output_path, *, title):
    raw_arr = coords_to_array(raw_coords)
    event_arr = coords_to_array(ransac_result["event"])
    smooth_arr = coords_to_array(ransac_result["smooth"])
    frame_axis = np.arange(len(raw_arr))
    statuses = np.asarray(ransac_result["statuses"], dtype=object)
    corrected = statuses == "outlier_corrected"
    ambiguous = statuses == "ambiguous_raw_kept"
    breakpoints = np.asarray(ransac_result["breakpoint_frames"], dtype=bool)

    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True, constrained_layout=True)
    coordinate_names = [(0, "x(frame)", "x coordinate (pixel)"), (1, "y(frame)", "y coordinate (pixel)")]
    for axis, (dimension, function_name, ylabel) in zip(axes, coordinate_names):
        axis.scatter(frame_axis, raw_arr[:, dimension], s=7, color="0.55", alpha=0.35, label="raw", zorder=1)
        axis.plot(frame_axis, event_arr[:, dimension], color="tab:orange", linewidth=1.0, label="ransac_event", zorder=2)
        axis.plot(frame_axis, smooth_arr[:, dimension], color="tab:blue", linewidth=1.4, label="ransac_smooth", zorder=3)
        axis.scatter(frame_axis[corrected], event_arr[corrected, dimension], s=24, marker="x", color="tab:red", label="corrected", zorder=4)
        axis.scatter(frame_axis[ambiguous], event_arr[ambiguous, dimension], s=24, marker="D", facecolors="none", edgecolors="tab:purple", label="ambiguous kept", zorder=4)
        axis.scatter(frame_axis[breakpoints], event_arr[breakpoints, dimension], s=42, marker="P", color="tab:green", label="breakpoint kept", zorder=5)
        axis.set_title(function_name)
        axis.set_ylabel(ylabel)
        axis.grid(alpha=0.22)
        axis.legend(loc="best", ncols=5, fontsize=8)

    axes[-1].set_xlabel("frame")
    fig.suptitle(title)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    return output_path


OUTPUT_PREFIX = metadata.get("output_prefix") or MATCH_FRAMES_PATH.stem
XY_FUNCTION_PLOT_PATH = CV_OUTPUT_DIR / f"{OUTPUT_PREFIX}_xy_function_coordinates.png"
save_xy_function_plot(
    raw_coords,
    ransac,
    XY_FUNCTION_PLOT_PATH,
    title=f"{OUTPUT_PREFIX}: trajectory coordinates by frame",
)

In [ ]:
plot_court_projection(
    {
        "raw": raw_coords,
        "current": current_coords,
        "optimized_smooth": optimized["smooth"],
        "ransac_event": ransac["event"],
        "ransac_smooth": ransac["smooth"],
    },
    start=START,
    end=END,
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
frames_axis = np.arange(len(raw_coords))
axes[0].plot(frames_axis, ransac["residuals"], linewidth=0.8, label="RANSAC residual")
axes[0].plot(frames_axis, ransac["thresholds"], linewidth=1.0, label="adaptive threshold")
axes[0].set_ylabel("px")
axes[0].set_title("Window RANSAC residual vs adaptive threshold")
axes[0].legend()
axes[0].grid(alpha=0.25)

status_colors = {
    "breakpoint_raw_kept": "tab:green",
    "breakpoint_gap": "tab:olive",
    "outlier_corrected": "tab:red",
    "ambiguous_raw_kept": "tab:orange",
    "suspect_recovered_raw_kept": "tab:purple",
    "short_gap_interp": "tab:green",
    "inlier_raw_kept": "tab:blue",
    "raw_kept": "tab:gray",
}
for status, color in status_colors.items():
    mask = status_df.status == status
    axes[1].scatter(status_df.loc[mask, "frame"], np.full(mask.sum(), status), s=8, c=color, label=status)
axes[1].set_xlabel("frame")
axes[1].set_title("RANSAC point status")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].grid(alpha=0.25)
plt.show()

## 6. Bounce Candidate Compare

For bounce detection, test both `optimized_event` and `optimized_smooth`. My starting bias: use event trajectory for candidate detection, and smoothed trajectory only for downstream speed / drawing.

In [ ]:
def detect_bounces(coords, *, score_thresh=15.0, half_window=5):
    detector = BounceDetectorManager(score_thresh=score_thresh)
    bounces, scores, angles = detector.detect(coords, half_window=half_window)
    enhanced = detector.enhance_subpixel(coords, bounces)
    return enhanced, scores, angles


current_bounces, current_scores, current_angles = detect_bounces(current_coords, score_thresh=15.0)
event_bounces, event_scores, event_angles = detect_bounces(optimized["event"], score_thresh=15.0)
smooth_bounces, smooth_scores, smooth_angles = detect_bounces(optimized["smooth"], score_thresh=15.0)
ransac_event_bounces, ransac_event_scores, ransac_event_angles = detect_bounces(ransac["event"], score_thresh=15.0)
ransac_smooth_bounces, ransac_smooth_scores, ransac_smooth_angles = detect_bounces(ransac["smooth"], score_thresh=15.0)

pd.DataFrame([
    {"track": "current", "bounce_count": len(current_bounces), "frames": [b["frame"] for b in current_bounces[:20]]},
    {"track": "optimized_event", "bounce_count": len(event_bounces), "frames": [b["frame"] for b in event_bounces[:20]]},
    {"track": "optimized_smooth", "bounce_count": len(smooth_bounces), "frames": [b["frame"] for b in smooth_bounces[:20]]},
    {"track": "ransac_event", "bounce_count": len(ransac_event_bounces), "frames": [b["frame"] for b in ransac_event_bounces[:20]]},
    {"track": "ransac_smooth", "bounce_count": len(ransac_smooth_bounces), "frames": [b["frame"] for b in ransac_smooth_bounces[:20]]},
])

In [ ]:
def bounce_df(name, bounces):
    return pd.DataFrame([
        {
            "track": name,
            "frame": b.get("frame"),
            "x": float(b.get("coord", [np.nan, np.nan])[0]),
            "y": float(b.get("coord", [np.nan, np.nan])[1]),
            "sub_coord": b.get("sub_coord"),
            "score": round(float(b.get("score", np.nan)), 2),
            "angle": round(float(b.get("angle", np.nan)), 2),
        }
        for b in bounces
    ])


bounces_df = pd.concat([
    bounce_df("current", current_bounces),
    bounce_df("optimized_event", event_bounces),
    bounce_df("optimized_smooth", smooth_bounces),
    bounce_df("ransac_event", ransac_event_bounces),
    bounce_df("ransac_smooth", ransac_smooth_bounces),
], ignore_index=True)

bounces_df[(bounces_df.frame >= START) & (bounces_df.frame < END)].sort_values(["frame", "track"])

## 7. Parameter Sweep

Small grid search for sanity. This is not a replacement for labeled evaluation, but it quickly shows which parameters explode bounce count or remove too many points.

In [ ]:
rows = []
for speed_scale in [4.0, 5.0, 6.0, 8.0]:
    for accel_scale in [5.0, 7.0, 9.0]:
        cfg = TrajectoryConfig(speed_mad_scale=speed_scale, accel_mad_scale=accel_scale, smooth_window=7)
        result = optimized_process(raw_coords, cfg)
        bounces, _, _ = detect_bounces(result["event"], score_thresh=15.0)
        stats = trajectory_stats("candidate", result["event"])
        rows.append({
            "speed_mad_scale": speed_scale,
            "accel_mad_scale": accel_scale,
            "removed_points": result["info"]["removed_points"],
            "speed_threshold": result["info"]["speed_threshold"],
            "accel_threshold": result["info"]["accel_threshold"],
            "valid_ratio": stats["valid_ratio"],
            "p95_speed": stats["p95_speed_px_frame"],
            "bounce_count": len(bounces),
        })

sweep_df = pd.DataFrame(rows).sort_values(["bounce_count", "removed_points"])
sweep_df

## 8. Render RANSAC Overlay Video

This writes the RANSAC event/smooth points back onto the source video for visual inspection.

In [ ]:
import cv2


def resolve_source_video_path(metadata):
    candidates = []
    for key in ["source_video_path", "video_path", "input_video_path"]:
        value = metadata.get(key)
        if value:
            path = Path(value).expanduser()
            candidates.append(path if path.is_absolute() else PROJECT_ROOT / path)

    source_name = metadata.get("source_video")
    if source_name:
        candidates.extend([
            INPUT_DIR / source_name,
            CV_OUTPUT_DIR / source_name,
            MATCH_FRAMES_PATH.parent / source_name,
        ])

    for path in candidates:
        if path.exists():
            return path
    return None


def _point_from_arr(arr, index, scale_x=1.0, scale_y=1.0):
    if index is None or index < 0 or index >= len(arr):
        return None
    x, y = arr[index]
    if not np.isfinite(x) or not np.isfinite(y):
        return None
    return int(round(x * scale_x)), int(round(y * scale_y))


def _draw_tail(frame, arr, center_index, *, scale_x, scale_y, tail_len, color, thickness=2):
    start = max(0, center_index - tail_len)
    previous = None
    for index in range(start, center_index + 1):
        point = _point_from_arr(arr, index, scale_x, scale_y)
        if point is None:
            previous = None
            continue
        if previous is not None:
            cv2.line(frame, previous, point, color, thickness, cv2.LINE_AA)
        previous = point


def _draw_status_marker(frame, point, status):
    if point is None:
        return
    x, y = point
    if status == "outlier_corrected":
        cv2.line(frame, (x - 10, y - 10), (x + 10, y + 10), (40, 40, 255), 2, cv2.LINE_AA)
        cv2.line(frame, (x - 10, y + 10), (x + 10, y - 10), (40, 40, 255), 2, cv2.LINE_AA)
    elif status == "short_gap_interp":
        cv2.circle(frame, point, 10, (70, 220, 70), 2, cv2.LINE_AA)
    elif status == "inlier_raw_kept":
        cv2.circle(frame, point, 8, (0, 210, 255), 2, cv2.LINE_AA)
    elif status == "ambiguous_raw_kept":
        cv2.rectangle(frame, (x - 8, y - 8), (x + 8, y + 8), (0, 165, 255), 2, cv2.LINE_AA)
    elif status == "suspect_recovered_raw_kept":
        cv2.circle(frame, point, 9, (180, 80, 220), 2, cv2.LINE_AA)
    elif status == "breakpoint_raw_kept":
        cv2.drawMarker(frame, point, (255, 0, 255), cv2.MARKER_DIAMOND, 20, 2, cv2.LINE_AA)
    elif status == "raw_kept":
        cv2.circle(frame, point, 7, (180, 180, 180), 2, cv2.LINE_AA)


def _put_hud(frame, lines):
    overlay = frame.copy()
    height = 30 + 24 * len(lines)
    cv2.rectangle(overlay, (12, 12), (620, height), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
    for row, text in enumerate(lines):
        cv2.putText(
            frame,
            text,
            (24, 42 + 24 * row),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.62,
            (245, 245, 245),
            1,
            cv2.LINE_AA,
        )


def render_ransac_overlay_video(
    input_video_path,
    output_video_path,
    raw_coords,
    ransac_result,
    *,
    frame_indices=None,
    confidences=None,
    tail_len=24,
):
    input_video_path = Path(input_video_path)
    output_video_path = Path(output_video_path)
    output_video_path.parent.mkdir(parents=True, exist_ok=True)

    if not input_video_path.exists():
        raise FileNotFoundError(f"Source video not found: {input_video_path}")

    raw_arr = coords_to_array(raw_coords)
    event_arr = coords_to_array(ransac_result["event"])
    smooth_arr = coords_to_array(ransac_result["smooth"])
    statuses = list(ransac_result.get("statuses", []))
    residuals = np.asarray(ransac_result.get("residuals", []), dtype=np.float64)
    thresholds = np.asarray(ransac_result.get("thresholds", []), dtype=np.float64)
    confidences_arr = np.asarray(confidences if confidences is not None else [], dtype=np.float64)

    if frame_indices is None:
        frame_indices = list(range(len(event_arr)))
    frame_indices = [int(frame) for frame in frame_indices]
    if len(frame_indices) != len(event_arr):
        raise ValueError("frame_indices length must match the RANSAC track length.")

    track_index_by_video_frame = {frame_number: index for index, frame_number in enumerate(frame_indices)}
    last_video_frame = max(frame_indices) if frame_indices else -1

    cap = cv2.VideoCapture(str(input_video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open source video: {input_video_path}")

    source_fps = cap.get(cv2.CAP_PROP_FPS) or fps
    source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or frame_width)
    source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or frame_height)
    scale_x = source_width / max(frame_width, 1)
    scale_y = source_height / max(frame_height, 1)

    writer = cv2.VideoWriter(
        str(output_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        source_fps,
        (source_width, source_height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create output video: {output_video_path}")

    try:
        from tqdm.auto import tqdm
        progress = tqdm(total=last_video_frame + 1, desc="render ransac overlay")
    except Exception:
        progress = None

    written = 0
    video_frame_index = 0
    try:
        while video_frame_index <= last_video_frame:
            ok, frame = cap.read()
            if not ok:
                break

            track_index = track_index_by_video_frame.get(video_frame_index)
            if track_index is not None:
                raw_point = _point_from_arr(raw_arr, track_index, scale_x, scale_y)
                event_point = _point_from_arr(event_arr, track_index, scale_x, scale_y)
                smooth_point = _point_from_arr(smooth_arr, track_index, scale_x, scale_y)
                status = statuses[track_index] if track_index < len(statuses) else "unknown"

                _draw_tail(frame, smooth_arr, track_index, scale_x=scale_x, scale_y=scale_y, tail_len=tail_len, color=(255, 210, 40), thickness=2)
                _draw_tail(frame, event_arr, track_index, scale_x=scale_x, scale_y=scale_y, tail_len=tail_len, color=(0, 180, 255), thickness=2)

                if raw_point is not None:
                    cv2.circle(frame, raw_point, 2, (255, 0, 0), -1, cv2.LINE_AA)
                if smooth_point is not None:
                    cv2.circle(frame, smooth_point, 3, (255, 220, 40), 2, cv2.LINE_AA)
                if event_point is not None:
                    cv2.circle(frame, event_point, 1, (0, 210, 255), -1, cv2.LINE_AA)
                    _draw_status_marker(frame, event_point, status)
                if status == "outlier_corrected" and raw_point is not None and event_point is not None:
                    cv2.line(frame, raw_point, event_point, (40, 40, 255), 1, cv2.LINE_AA)

                conf = confidences_arr[track_index] if track_index < len(confidences_arr) else np.nan
                residual = residuals[track_index] if track_index < len(residuals) else np.nan
                threshold = thresholds[track_index] if track_index < len(thresholds) else np.nan
                hud_lines = [
                    f"video frame: {video_frame_index} | track row: {track_index}",
                    f"status: {status} | conf: {conf:.3f}" if np.isfinite(conf) else f"status: {status} | conf: n/a",
                    f"residual/thresh: {residual:.1f}/{threshold:.1f} px" if np.isfinite(residual) and np.isfinite(threshold) else "residual/thresh: n/a",
                    "blue raw | orange event | cyan smooth | magenta breakpoint",
                ]
                _put_hud(frame, hud_lines)

            writer.write(frame)
            written += 1
            video_frame_index += 1
            if progress is not None:
                progress.update(1)
    finally:
        if progress is not None:
            progress.close()
        cap.release()
        writer.release()

    return {
        "input_video": str(input_video_path),
        "output_video": str(output_video_path),
        "frames_written": written,
        "source_fps": source_fps,
        "source_size": (source_width, source_height),
    }


RANSAC_SOURCE_VIDEO_PATH = resolve_source_video_path(metadata)
RANSAC_OVERLAY_VIDEO_PATH = CV_OUTPUT_DIR / f"{metadata.get('output_prefix') or MATCH_FRAMES_PATH.stem}_ransac_trajectory_overlay.mp4"
FRAME_INDICES = [int(frame.get("frame_index", index)) for index, frame in enumerate(frames)]

if RANSAC_SOURCE_VIDEO_PATH is None:
    raise FileNotFoundError(
        "Could not infer the source video path from metadata. "
        "Set RANSAC_SOURCE_VIDEO_PATH manually before running this cell."
    )

render_ransac_overlay_video(
    RANSAC_SOURCE_VIDEO_PATH,
    RANSAC_OVERLAY_VIDEO_PATH,
    raw_coords,
    ransac,
    frame_indices=FRAME_INDICES,
    confidences=raw_confidences,
    tail_len=24,
)


## 9. Export Candidate Track

This writes a compact CSV for quick diffing or visualization elsewhere.

In [ ]:
EXPORT_PATH = CV_OUTPUT_DIR / "trajectory_optimization_candidate.csv"

raw_arr = coords_to_array(raw_coords)
current_arr = coords_to_array(current_coords)
event_arr = coords_to_array(optimized["event"])
smooth_arr = coords_to_array(optimized["smooth"])
ransac_event_arr = coords_to_array(ransac["event"])
ransac_smooth_arr = coords_to_array(ransac["smooth"])

export_df = pd.DataFrame({
    "frame": np.arange(len(raw_coords)),
    "raw_x": raw_arr[:, 0],
    "raw_y": raw_arr[:, 1],
    "current_x": current_arr[:, 0],
    "current_y": current_arr[:, 1],
    "optimized_event_x": event_arr[:, 0],
    "optimized_event_y": event_arr[:, 1],
    "optimized_smooth_x": smooth_arr[:, 0],
    "optimized_smooth_y": smooth_arr[:, 1],
    "ransac_event_x": ransac_event_arr[:, 0],
    "ransac_event_y": ransac_event_arr[:, 1],
    "ransac_smooth_x": ransac_smooth_arr[:, 0],
    "ransac_smooth_y": ransac_smooth_arr[:, 1],
    "ransac_status": ransac["statuses"],
    "ransac_initial_residual_px": ransac["initial_residuals"],
    "ransac_residual_px": ransac["residuals"],
    "ransac_threshold_px": ransac["thresholds"],
    "ransac_suspect": ransac["suspect_mask"],
    "ransac_breakpoint_protected": ransac["breakpoint_keep_mask"],
    "ransac_breakpoint_event_run": ransac["breakpoint_run_mask"],
    "ransac_breakpoint_frame": ransac["breakpoint_frames"],
    "ransac_breakpoint_angle_deg": ransac["breakpoint_angles"],
    "ransac_trusted_model": ransac["trusted_models"],
    "ransac_left_inliers": ransac["left_inliers"],
    "ransac_right_inliers": ransac["right_inliers"],
})
export_df.to_csv(EXPORT_PATH, index=False)
EXPORT_PATH

## Notes For Backporting

If the optimized behavior looks better, the clean backport path is:

- Add config fields to `BallTrackerManager.process_trajectory` or create a new `process_trajectory_v2`.
- Return both `event_coords` and `smooth_coords`, or add a small data object with named tracks.
- Use `event_coords` for bounce/hit candidates.
- Use `smooth_coords` for drawing and speed-style metrics.
- Fix TrackNet postprocess scaling before judging any final threshold.